# SWEAT tutorials: EVASPA Evaporative Fraction model and configuration

Learn how to configure EF model

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pprint import pprint

In [ ]:
from sweat.evaspa.ef import (
    EFConfig,
    get_variables_from_models,
    initialize,
    run,
)

In [ ]:
from sweat.extra.plot_notebook import plot_dataset, plot_edges, plot_models

In [ ]:
from extra.prepare import setup_data

### Generate test data 

Generate synthetic data

In [ ]:
# Generate data
data = setup_data(
    albedo=(0.0, 0.6),
    fcover=(0.0, 1.0),
    dry=(330.0, -10.0),
    wet=(300.0, 15.0),
    valid=(0.2, 0.8),
)

In [ ]:
data

In [ ]:
plot_dataset(data)

In [ ]:
plot_edges(data["albedo"], data["lst"])

In [ ]:
plot_edges(data["fcover"], data["lst"])

### Define configuration

Define a list of EF models. The configuration of EF model can be found [here](https://github.com/CNES/sweat/blob/main/docs/evaspa/ef_model.md)

In [ ]:
config = {
    "models": [
        {
            "name": "model1",
            "dry_edge": {
                "type": "LinearEdge",
                "config": {
                    "interval_type": "size",
                    "interval_nb": 20,
                    "percentile": 2,
                    "selection": "median",
                },
            },
            "wet_edge": {
                "type": "LinearEdge",
                "config": {
                    "interval_type": "size",
                    "interval_nb": 20,
                    "percentile": 2,
                    "selection": "median",
                },
            },
            "var": "fcover",
        },
        {
            "name": "model2",
            "dry_edge": {
                "type": "LinearEdge",
                "config": {
                    "interval_type": "density",
                    "interval_nb": 20,
                    "percentile": 5,
                    "selection": "median",
                },
            },
            "wet_edge": {
                "type": "FlatEdge",
            },
            "var": "albedo",
        },
    ],
    "options": {
        "filtering": {"lst": {"op": ">", "value": 310}},
        "selection": False,
        "merging": {
            "merging_method": "mean",
            "uncertainty_method": "interquartile",
        },
    },
}

In [ ]:
pprint(config, sort_dicts=False)

### Check configuration

In [ ]:
# Validate parameters config
ef_config = EFConfig.model_validate(config).model_dump()

*If an error occurs in the previous cell, it means that there is an error in the definition of the EF models.*

In [ ]:
pprint(ef_config, sort_dicts=False)

### Run

Initialize

In [ ]:
models, options = initialize(config)

In [ ]:
get_variables_from_models(models=models)

In [ ]:
models

In [ ]:
options

Compute

In [ ]:
ef, ef_inst = run(models, data, **options)

In [ ]:
ef

In [ ]:
ef["model2"].attrs

In [ ]:
ef_inst

Visualize: Plot the EF models

In [ ]:
plot_models(data, models=ef)